# 04 — Model Evaluation

Evaluate all trained models with classification reports, confusion matrices,
and side-by-side comparison. Save results for the dashboard.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')

## 1. Load Data & Models

In [ ]:
from src.model import load_model, evaluate_model
from src.preprocessing import RISK_LABEL_MAP

processed = Path('../data/processed')
outputs = Path('../outputs')
outputs.mkdir(exist_ok=True)

# Load test data
X_test_d = pd.read_csv(processed / 'diabetes_X_test.csv')
y_test_d = pd.read_csv(processed / 'diabetes_y_test.csv').squeeze()
X_test_h = pd.read_csv(processed / 'heart_X_test.csv')
y_test_h = pd.read_csv(processed / 'heart_y_test.csv').squeeze()

model_names = ['logistic_regression', 'random_forest', 'xgboost', 'lightgbm']

## 2. Evaluate — Diabetes

In [ ]:
diabetes_evals = {}
for name in model_names:
    try:
        model = load_model(name, 'diabetes')
        result = evaluate_model(model, X_test_d, y_test_d)
        diabetes_evals[name] = result
        print(f'\n=== {name.upper()} ===')
        print(f'F1-Macro: {result["f1_macro"]:.4f}')
        print(f'ROC-AUC:  {result["roc_auc"]:.4f}' if result['roc_auc'] else 'ROC-AUC: N/A')
    except FileNotFoundError:
        print(f'{name}: not trained yet, skipping')

### 2.1 Confusion Matrices — Diabetes

In [ ]:
from src.utils import plot_confusion_matrix, save_plot

labels = ['Low', 'Medium', 'High']
n_models = len(diabetes_evals)
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, (name, result) in zip(axes, diabetes_evals.items()):
    sns.heatmap(result['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f'{name}\nF1={result["f1_macro"]:.3f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Diabetes — Confusion Matrices', fontsize=14)
plt.tight_layout()
save_plot(fig, 'diabetes_confusion_matrices.png')
plt.show()

## 3. Evaluate — Heart Disease

In [ ]:
heart_evals = {}
for name in model_names:
    try:
        model = load_model(name, 'heart')
        result = evaluate_model(model, X_test_h, y_test_h)
        heart_evals[name] = result
        print(f'\n=== {name.upper()} ===')
        print(f'F1-Macro: {result["f1_macro"]:.4f}')
        print(f'ROC-AUC:  {result["roc_auc"]:.4f}' if result['roc_auc'] else 'ROC-AUC: N/A')
    except FileNotFoundError:
        print(f'{name}: not trained yet, skipping')

### 3.1 Confusion Matrices — Heart Disease

In [ ]:
n_models = len(heart_evals)
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, (name, result) in zip(axes, heart_evals.items()):
    sns.heatmap(result['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f'{name}\nF1={result["f1_macro"]:.3f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Heart Disease — Confusion Matrices', fontsize=14)
plt.tight_layout()
save_plot(fig, 'heart_confusion_matrices.png')
plt.show()

## 4. Model Comparison Summary

In [ ]:
def build_comparison(evals, dataset_name):
    rows = []
    for name, result in evals.items():
        rows.append({
            'Model': name,
            'F1-Macro': round(result['f1_macro'], 4),
            'ROC-AUC': round(result['roc_auc'], 4) if result['roc_auc'] else None,
            'Precision (macro)': round(result['classification_report']['macro avg']['precision'], 4),
            'Recall (macro)': round(result['classification_report']['macro avg']['recall'], 4),
        })
    df = pd.DataFrame(rows).sort_values('F1-Macro', ascending=False).reset_index(drop=True)
    df.to_csv(outputs / f'{dataset_name}_model_comparison.csv', index=False)
    return df

print('=== DIABETES ===')
diab_comp = build_comparison(diabetes_evals, 'diabetes')
print(diab_comp.to_string(index=False))

print('\n=== HEART DISEASE ===')
heart_comp = build_comparison(heart_evals, 'heart')
print(heart_comp.to_string(index=False))

## 5. Comparison Charts

In [ ]:
from src.utils import plot_model_comparison, save_plot

fig = plot_model_comparison(diab_comp)
fig.suptitle('Diabetes — Model Comparison')
save_plot(fig, 'diabetes_model_comparison.png')
plt.show()

fig = plot_model_comparison(heart_comp)
fig.suptitle('Heart Disease — Model Comparison')
save_plot(fig, 'heart_model_comparison.png')
plt.show()

print('Comparison results saved to outputs/')